# 18 · Why validation did not carry over to test (E18, diagnosis)

E17 scored the 5 saved E16 agents (Double DQN, state A: market factors only) once on test and stress_bear. Their
validation Sharpe (0.88 - 1.61) did not predict their test Sharpe (−6.85 - +1.45). This notebook looks at **what
each agent did** on train, validation, test and stress_bear, to explain that gap:

1. **Behaviour**: how much of the time each agent is long / short / flat, how often it trades, how long it holds.
2. **Where the return came from**: average direction ("exposure") vs changes of position ("timing") vs costs.
3. **Timing skill**: does the position at the start of a day predict that day's market move?
4. **Decision margin**: how close the best and second-best action's Q-values are (small margin = fragile choice).
5. **Input shift**: how different the 8 market factors are on each split from what the agent was trained on.
6. **Seed agreement**: do the 5 seeds hold the same position at the same time?
7. **Selection noise**: how precisely one validation period measures a policy's Sharpe.

**Nothing here is trained, tuned or chosen.** The agents, checkpoints and results are the fixed E16 / E17 ones.
Test and stress_bear were already scored once in E17. This notebook reads them only to explain those results, so
any change made because of what it shows has to be judged on data not used so far, not on test or stress_bear.

Writes `thesis_rl/initiation/e18_diagnosis/` (per-step traces and tables) and `report/e18_*`. CPU runtime is
enough (about 15-30 minutes, mostly the train split). A second run reuses the saved traces.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
seeding.record_environment(L.report / "environment", "18_initiation_diagnosis")

In [ ]:
import json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from rltrade import diagnose as D, results as R
from rltrade.runner import RunSpec
from rltrade.env import EnvConfig
from rltrade.metrics import compute_metrics

INIT = L.root / "initiation"
AG, REP, DIAG = INIT / "agents", INIT / "report", INIT / "e18_diagnosis"
TR = DIAG / "traces"
for p in (REP, TR): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
SPLITS = ("train", "val", "test", "stress_bear")
NEEDED = [f"fixed_plus/market_{s}.npz" for s in SPLITS] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)
    print("unzipped", NEEDED, "->", LOCAL)

sel = json.loads((INIT / "selected_agent_e16.json").read_text())
def on_drive(p):                                    # paths written on another machine: same place under initiation/
    p = Path(p)
    return p if p.exists() else INIT.joinpath(*p.parts[p.parts.index("initiation") + 1:])
F = on_drive(sel["factor_dir"])
AGENTS = list(sel["runs"])
BH = RunSpec("16_agents", "buy_and_hold", env=EnvConfig(state="A")).run_id
NAME = {r: "seed " + r.rsplit("_s", 1)[1] for r in AGENTS} | {BH: "buy and hold"}
STEP_GAP = 5
PPY = 365 * D.steps_per_day(STEP_GAP)
val = R.load_results(AG).set_index("run_id")
E17 = {s: pd.read_csv(INIT / "e17_final" / f"{s}_runs.csv").set_index("run_id") for s in ("test", "stress_bear")}
print(f"{len(AGENTS)} agents + buy and hold | splits {SPLITS} | factors {F}")

## Traces: one exact backtest per agent and split, with Q-values

In [ ]:
def get_trace(rid, split):
    f = TR / f"{rid.split('/')[-1]}_{split}.npz"
    if f.exists():
        return dict(np.load(f))
    tr = D.trace(AG / "models" / rid, split, PREP, F)
    np.savez_compressed(f, **{k: tr[k] for k in D.TRACE_KEYS})
    return tr

T, t0 = {}, time.time()
for rid in AGENTS + [BH]:
    for s in SPLITS:
        T[rid, s] = get_trace(rid, s)
    n_steps = ", ".join(f"{s} {len(T[rid, s]['level']):,}" for s in SPLITS)
    print(f"  {NAME[rid]}: traced {n_steps} steps ({time.time() - t0:.0f} s)")

# the traces must reproduce the numbers already reported (E16 validation, E17 test / stress_bear)
chk = []
for rid in AGENTS + [BH]:
    for s in ("val", "test", "stress_bear"):
        got = compute_metrics(T[rid, s], PPY)["sharpe"]
        ref = val.loc[rid, "val_sharpe"] if s == "val" else E17[s].loc[rid, "sharpe"]
        chk.append({"policy": NAME[rid], "split": s, "reported": ref, "trace": got, "diff": abs(got - ref)})
chk = pd.DataFrame(chk)
print("largest difference from the reported Sharpe:", f"{chk['diff'].max():.2e}")
if chk["diff"].max() > 1e-3:
    print("WARNING - some traces differ from the reported numbers (e.g. GPU vs CPU arithmetic):")
    print(chk[chk["diff"] > 1e-3].round(4).to_string(index=False))

## 0 · The market on each split

In [ ]:
mk = []
for s in SPLITS:
    mr = np.asarray(T[BH, s]["mret"], float); spd = D.steps_per_day(STEP_GAP); n = len(mr) // spd
    day = np.log1p(mr[: n * spd]).reshape(n, spd).sum(1)
    mk.append({"split": s, "days": n, "BTC return %": 100 * (np.prod(1 + mr) - 1),
               "annual vol %": 100 * mr.std() * np.sqrt(PPY), "up days %": 100 * np.mean(day > 0),
               "buy and hold Sharpe": compute_metrics(T[BH, s], PPY)["sharpe"]})
market = pd.DataFrame(mk)
print(market.round(2).to_string(index=False))

## 1 · Behaviour

In [ ]:
rows = []
for s in SPLITS:
    for rid in AGENTS:
        tr = T[rid, s]
        rows.append({"split": s, "policy": NAME[rid], "Sharpe": compute_metrics(tr, PPY)["sharpe"], **D.behaviour(tr, STEP_GAP)})
beh = pd.DataFrame(rows)
beh.to_csv(DIAG / "behaviour.csv", index=False)
print(beh.round(2).to_string(index=False))
print("\nheld_capped_pct = share of decisions where the 'time in position' input is already at its cap (1 day)")

## 2 · Where the return came from (sums of step returns, %)

In [ ]:
rows = []
for s in SPLITS:
    for rid in AGENTS + [BH]:
        rows.append({"split": s, "policy": NAME[rid], **D.attribution(T[rid, s], STEP_GAP)})
att = pd.DataFrame(rows)
att.to_csv(DIAG / "attribution.csv", index=False)
cols = ["split", "policy", "net_pct", "exposure_pct", "timing_pct", "costs_pct", "from_long_pct", "from_short_pct",
        "mean_level", "beta", "alpha_ann_pct", "r2"]
print(att[cols].round(2).to_string(index=False))
print("\nexposure = a constant position at the agent's average level; timing = what changing position added;"
      "\nbeta / alpha / r2 = regression of the agent's step returns on BTC's (alpha annualised)."
      f"\nlargest |residual| (net - gross + costs): {att['residual_pct'].abs().max():.3f} %")

## 3 · Timing skill: position at the start of a day vs that day's BTC return

In [ ]:
rows = []
for s in SPLITS:
    for rid in AGENTS:
        rows.append({"split": s, "policy": NAME[rid], **D.daily_timing(T[rid, s], STEP_GAP)})
tim = pd.DataFrame(rows)
tim.to_csv(DIAG / "timing.csv", index=False)
print(tim.round(3).to_string(index=False))
print("\nic = Spearman correlation (NaN: the position never changed between days); hit = share of non-flat days"
      "\nwhose direction matched the day's move.")

## 4 · Decision margin (Q-value gap between the best and second-best action)

In [ ]:
rows, GAP = [], {}
QAG = [r for r in AGENTS if len(D.q_gap(T[r, "train"]))]     # runs that trade with their network (no fallback)
for rid in QAG:
    ref = np.median(D.q_gap(T[rid, "train"]))
    for s in SPLITS:
        g = D.q_gap(T[rid, s]); GAP[rid, s] = g / ref
        rows.append({"split": s, "policy": NAME[rid], "median gap / train median": np.median(g) / ref,
                     "steps with gap < 10 % of train median %": 100 * np.mean(g < 0.1 * ref),
                     "trades per day": D.behaviour(T[rid, s], STEP_GAP)["trades_per_day"]})
qg = pd.DataFrame(rows)
qg.to_csv(DIAG / "q_gap.csv", index=False)
print(qg.round(3).to_string(index=False))

## 5 · Input shift: the 8 market factors on each split vs train

In [ ]:
FAC = {s: np.load(F / f"factors_{s}.npy") for s in SPLITS}
rows, per_factor = [], []
for s in SPLITS[1:]:
    fr, any_out = D.factor_shift(FAC["train"], FAC[s])
    per_factor += [{"split": s, **r} for r in fr]
    rows.append({"split": s, "mean KS": np.mean([r["ks"] for r in fr]), "max KS": max(r["ks"] for r in fr),
                 "max |mean shift| (train sd)": max(abs(r["mean_shift_sd"]) for r in fr),
                 "rows with a factor outside train's 0.5-99.5 % range %": any_out})
shift = pd.DataFrame(rows); per_factor = pd.DataFrame(per_factor)
per_factor.to_csv(DIAG / "factor_shift.csv", index=False)
print(shift.round(3).to_string(index=False))
print("\nper factor:\n" + per_factor.pivot(index="factor", columns="split", values="ks").round(3).to_string())

## 6 · Do the seeds agree?

In [ ]:
rows = []
for s in SPLITS:
    rows.append({"split": s, **D.agreement(np.stack([T[r, s]["level"] for r in AGENTS]))})
agr = pd.DataFrame(rows)
agr.to_csv(DIAG / "agreement.csv", index=False)
print(agr.round(1).to_string(index=False))
print("\npairwise_same = share of steps two seeds hold the same sign (long / short / flat), averaged over pairs")

## 7 · Selection noise: how precisely does one validation period measure a policy?

In [ ]:
rows = []
for rid in AGENTS:
    bs = D.bootstrap_sharpe(T[rid, "val"]["returns"], T[BH, "val"]["returns"], STEP_GAP)
    vc = json.loads((AG / "models" / rid / "history.json").read_text())["val_curve"]
    ch = [c["sharpe"] for c in vc]
    rows.append({"policy": NAME[rid], "val Sharpe": val.loc[rid, "val_sharpe"],
                 "val 5 %": bs["sharpe_p5"], "val 95 %": bs["sharpe_p95"],
                 "val resamples beating B&H %": bs["beats_ref_pct"],
                 "checkpoints": len(vc), "ckpt val Sharpe sd (screen)": np.std(ch, ddof=1),
                 "train Sharpe": compute_metrics(T[rid, "train"], PPY)["sharpe"],
                 "test Sharpe": E17["test"].loc[rid, "sharpe"], "stress Sharpe": E17["stress_bear"].loc[rid, "sharpe"]})
selt = pd.DataFrame(rows)
selt.to_csv(DIAG / "selection.csv", index=False)
print(selt.round(3).to_string(index=False))
rv_t = stats.spearmanr(selt["val Sharpe"], selt["test Sharpe"])
rv_s = stats.spearmanr(selt["val Sharpe"], selt["stress Sharpe"])
rt_v = stats.spearmanr(selt["train Sharpe"], selt["val Sharpe"])
sel_note = (f"rank correlation across the 5 seeds: val vs test {rv_t.statistic:+.2f}, val vs stress_bear "
            f"{rv_s.statistic:+.2f}, train vs val {rt_v.statistic:+.2f} (5 points: indicative only)")
print("\n" + sel_note)
bh_bs = D.bootstrap_sharpe(T[BH, "val"]["returns"], T[BH, "val"]["returns"], STEP_GAP)
print(f"buy and hold val Sharpe {val.loc[BH, 'val_sharpe']:+.2f}, 5-95 %: {bh_bs['sharpe_p5']:+.2f} to {bh_bs['sharpe_p95']:+.2f}")

## Figures

In [ ]:
COL = {1: "#2a6fdb", -1: "#e8743b", 0: "#d9d9d9"}          # long / short / flat
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
cmap = ListedColormap([COL[-1], COL[0], COL[1]])
fig, axes = plt.subplots(2, 4, figsize=(17, 5), gridspec_kw={"height_ratios": [1, 1.3]}, sharex="col")
for j, s in enumerate(SPLITS):
    t = R.as_datetime(T[BH, s]["time_ns"])
    axes[0, j].plot(t, np.cumprod(1 + np.asarray(T[BH, s]["mret"], float)), color="black", lw=1)
    axes[0, j].set_title(f"{s}: BTC (start = 1)", fontsize=10); axes[0, j].grid(alpha=0.3)
    every = 12                                              # one column per hour
    img = np.stack([np.sign(T[r, s]["level"][::every]) for r in AGENTS])
    axes[1, j].imshow(img, aspect="auto", cmap=cmap, vmin=-1, vmax=1, interpolation="nearest",
                      extent=[t[0], t[-1], len(AGENTS) - 0.5, -0.5])
    axes[1, j].set_yticks(range(len(AGENTS)), [NAME[r] for r in AGENTS], fontsize=8)
    axes[1, j].tick_params(axis="x", rotation=30, labelsize=8)
fig.legend(handles=[Patch(color=COL[1], label="long"), Patch(color=COL[-1], label="short"), Patch(color=COL[0], label="flat")],
           loc="lower center", ncol=3, fontsize=8, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1)); fig.savefig(REP / "e18_positions.png", dpi=130); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4), sharey=True)
for ax, s in zip(axes, SPLITS):
    ax.boxplot([np.log10(GAP[r, s] + 1e-6) for r in QAG], showfliers=False)
    ax.set_xticks(range(1, len(QAG) + 1), [NAME[r] for r in QAG], fontsize=8)
    ax.axhline(0, color="grey", lw=0.8, ls="--"); ax.set_title(s, fontsize=10); ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("log10(Q gap / train median)")
fig.suptitle("decision margin per step: lower = the chosen action barely beats the next one", fontsize=10)
fig.tight_layout(); fig.savefig(REP / "e18_qgap.png", dpi=130); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for r in AGENTS:
    vc = json.loads((AG / "models" / r / "history.json").read_text())["val_curve"]
    ax.plot([c["steps"] / 1e6 for c in vc], [c["sharpe"] for c in vc], lw=1.2, marker="o", ms=3, label=NAME[r])
ax.axhline(val.loc[BH, "val_sharpe"], color="black", ls="--", lw=1, label="buy and hold")
ax.set_xlabel("training steps (millions)"); ax.set_ylabel("validation Sharpe (screening pass)")
ax.set_title("validation Sharpe of every checkpoint", fontsize=10); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=3)
fig.tight_layout(); fig.savefig(REP / "e18_val_curves.png", dpi=130); plt.show()

In [ ]:
def tab(df, nd=2):
    return df.round(nd).to_markdown(index=False)
txt = ["# E18 diagnosis: why validation did not carry over (post-hoc; nothing trained or chosen)\n",
       "Agents: the 5 E16 Double DQN seeds (state A), fixed checkpoints. Test and stress_bear were scored once in E17; "
       "they are read here only to explain those results.\n",
       f"Traces reproduce the reported Sharpe to within {chk['diff'].max():.1e}.\n",
       "## 0 · Market\n", tab(market),
       "\n## 1 · Behaviour\n", tab(beh),
       "\n## 2 · Return attribution (sums of step returns, %)\n", tab(att[cols]),
       "\n## 3 · Daily timing\n", tab(tim, 3),
       "\n## 4 · Decision margin\n", tab(qg, 3),
       "\n## 5 · Factor shift vs train\n", tab(shift, 3), "", per_factor.pivot(index="factor", columns="split", values="ks").round(3).to_markdown(),
       "\n## 6 · Seed agreement\n", tab(agr, 1),
       "\n## 7 · Selection noise\n", tab(selt, 3), "", sel_note,
       f"\nbuy and hold val Sharpe {val.loc[BH, 'val_sharpe']:+.2f} (5-95 %: {bh_bs['sharpe_p5']:+.2f} to {bh_bs['sharpe_p95']:+.2f})\n",
       "Figures: e18_positions.png, e18_qgap.png, e18_val_curves.png"]
(REP / "e18_diagnosis.md").write_text("\n".join(txt) + "\n", encoding="utf-8")
print("wrote", REP / "e18_diagnosis.md", "|", DIAG)

In [ ]:
AUTO_DISCONNECT = True               # release the runtime once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()